In [1]:
import time
import pyvisa
from pyvisa import ResourceManager
import numpy as np
import os
import pandas as pd

In [2]:
OSC_ADDRESS =  "TCPIP0::192.168.1.200::INSTR"


In [3]:
_osc = None


def open_oscilloscope(address=None):
    """Open a VISA connection to the oscilloscope and cache it on this module."""
    global _osc
    # Use the pyvisa-py backend for consistent LAN/USB detection
    rm = pyvisa.ResourceManager('@py')
    target = address or OSC_ADDRESS
    _osc = rm.open_resource(target, timeout=5000)  # 5 seconds, adjust as needed
    return _osc


def close_oscilloscope():
    """Close the cached oscilloscope handle."""
    global _osc
    if _osc is not None:
        try:
            _osc.close()
        except Exception:
            pass
        _osc = None

In [4]:
def configure_oscilloscope_for_burst(osc, params):
    """Configure oscilloscope acquisition settings for burst waveform capture."""

    def vbs(cmd):
        osc.write(f"VBS '{cmd}'")
        time.sleep(0.1)

    freq = params["frequency"]
    amp = params["amplitude"]
    n_cycles = params.get("no_of_cycles_per_pulse", 1)

    burst_duration = n_cycles / freq
    hor_scale = burst_duration / 5
    ver_scale = amp
    sampling_rate = freq * 100

    vbs(f"app.Acquisition.Horizontal.HorScale = {hor_scale}")
    vbs(f"app.Acquisition.C1.VerScale = {ver_scale}")
    vbs(f"app.Acquisition.Horizontal.SampleRate = {sampling_rate}")
    vbs("app.Acquisition.C1.Offset = 0")
    vbs("app.Acquisition.C1.View = true")
    vbs('app.Acquisition.Trigger.Source = "C1"')
    osc.write("TRIG_MODE NORM")

In [ ]:
def read_oscilloscope_and_save(osc, SIGNAL_PARAMS, cross, s, scan_folder):
    """Read one waveform from the oscilloscope and save it as a CSV.

    Parameters
    ----------
    osc : pyvisa Resource | None
        Open oscilloscope handle.  Falls back to the module-level ``_osc``
        if *None* is passed.
    SIGNAL_PARAMS : dict
        Parameters for the signal being captured.
    cross, s : int
        Row and column indices used to name the output file.
    scan_folder : str
        Directory where the CSV will be written.
    """
    if osc is None:
        osc = _osc
    if osc is None:
        print("❌ Oscilloscope not open. Call open_oscilloscope() first.")
        return

    try:
        configure_oscilloscope_for_burst(osc, SIGNAL_PARAMS)

        osc.write("C1:WF? DAT1")
        raw_data = osc.query_binary_values(
            "C1:WF? DAT1", datatype="B", container=np.array
        )

        scale = float(osc.query("C1:VDIV?").strip().split(" ")[0])
        v_offset = float(osc.query("C1:OFST?").strip().split(" ")[0])
        scale1 = 1 / 30
        voltages = ((raw_data - 128) * scale + v_offset - 128) * scale1

        time_div = float(osc.query("TDIV?").strip().split(" ")[0])
        num_points = len(raw_data)
        time_span = 10 * time_div  # total span across 10 divisions
        time_values = np.linspace(0, time_span, num_points, endpoint=False)

        filename = f"row_{cross + 1}_col_{s}.csv"
        file_path = os.path.join(scan_folder, filename)
        df = pd.DataFrame({"Time (s)": time_values, "Amplitude (V)": voltages})
        df.to_csv(file_path, index=False)
        print(f"✅ Data saved to: {file_path}")

        # Save signal parameters alongside waveform data
        param_path = os.path.join(scan_folder, "wave_parameter.csv")
        param_df = pd.DataFrame(
            list(SIGNAL_PARAMS.items()), columns=["Parameter", "Value"]
        )
        param_df.to_csv(param_path, index=False)
        print(f"📝 Signal parameters saved to: {param_path}")

    except Exception as e:
        print(f"❌ Failed to read oscilloscope data: {e}")

In [15]:
osc = open_oscilloscope(OSC_ADDRESS)

In [16]:
params=dict(frequency=1000, amplitude=1.0, no_of_cycles_per_pulse=10)

In [17]:
configure_oscilloscope_for_burst(_osc, params)

In [18]:
read_oscilloscope_and_save(_osc, params, cross=0, s=0, scan_folder="./data/")

❌ Failed to read oscilloscope data: list index out of range


In [10]:
# configure_oscilloscope_for_burst(osc, params)

# cross=0 
# s=0
# scan_folder="./data/"

# osc.write("C1:WF? DAT1")
# raw_data = osc.query_binary_values(
#     "C1:WF? DAT1", datatype="B", container=np.array
# )

# scale = float(osc.query("C1:VDIV?").strip().split(" ")[0])
# v_offset = float(osc.query("C1:OFST?").strip().split(" ")[0])
# scale1 = 1 / 30
# voltages = ((raw_data - 128) * scale + v_offset - 128) * scale1

# time_div = float(osc.query("TDIV?").strip().split(" ")[0])
# num_points = len(raw_data)
# time_span = 10 * time_div  # total span across 10 divisions
# time_values = np.linspace(0, time_span, num_points, endpoint=False)

# filename = f"row_{cross + 1}_col_{s}.csv"
# file_path = os.path.join(scan_folder, filename)
# df = pd.DataFrame({"Time (s)": time_values, "Amplitude (V)": voltages})
# df.to_csv(file_path, index=False)
# print(f"✅ Data saved to: {file_path}")

# # Save signal parameters alongside waveform data
# param_path = os.path.join(scan_folder, "wave_parameter.csv")
# param_df = pd.DataFrame(
#     list(params.items()), columns=["Parameter", "Value"]
# )
# param_df.to_csv(param_path, index=False)
# print(f"📝 Signal parameters saved to: {param_path}")

In [11]:
read_oscilloscope_and_save(osc, params, cross=0, s=0, scan_folder="./data/")

❌ Failed to read oscilloscope data: list index out of range


In [12]:
import plotly.express as px

In [13]:
fig = px.line(df[0:10000], x='Time (s)', y='Amplitude (V)', title='LeCroy HDO6054 Signal')

# Enable zooming functionality
fig.update_layout(dragmode='zoom')
fig.show()

NameError: name 'df' is not defined